# 🕵️ Dia 17 — Do modelo de ameaças ao mapa de evidências: qual registro prova o quê

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 17* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-17.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sexta, 9/out. A hospedagem do site falso foi suspensa, e o canal "Denuncie um golpe" foi adiado uma semana. A Marta pergunta: *"Das oito ameaças, se uma acontecer no primeiro mês, a gente consegue provar? Com qual registro?"*

Rafa: *"Quatro colunas por ameaça. Onde você escrever 'não sei', isso é um achado."*

Neste notebook você vai:

1. **Transformar o fluxograma de prontidão** numa função que aponta a lacuna de cada ameaça.
2. **Calcular a retenção real** a partir de uma configuração de `logrotate`.
3. **Ranquear as ações do plano** pelo número de ameaças que cada uma fecha.
4. **Desenhar o log de auditoria** proposto e gravar uma linha de exemplo encadeada por hash.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. As quatro checagens, em código

O fluxograma do README pergunta, nesta ordem: o registro **existe**? É guardado por **tempo suficiente**? Fica **fora do host**? O **custodiante** é conhecido? A primeira resposta "não" é a lacuna.

In [ ]:
def checar(linha, retencao_minima_dias):
    """
    linha: dicionário com 'existe' (S/N/Parcial), 'retencao_dias' (int ou None),
           'fora_do_host' (bool) e 'custodiante' (texto ou "").
    Devolve (estado, lacuna).
    """
    if linha["existe"] == "N":
        return ("NÃO INVESTIGÁVEL", "Criar o registro")
    if linha["retencao_dias"] is None:
        return ("LACUNA", "Retenção desconhecida: perguntar ao custodiante")
    if linha["retencao_dias"] < retencao_minima_dias:
        return ("LACUNA", f"Aumentar a retenção ({linha['retencao_dias']} d < {retencao_minima_dias} d)")
    if not linha["fora_do_host"]:
        return ("LACUNA", "O root do servidor pode apagá-lo: enviar para fora do host")
    if not linha["custodiante"]:
        return ("LACUNA", "Descobrir quem guarda (muitas vezes um terceiro)")
    estado = "INVESTIGÁVEL" if linha["existe"] == "S" else "INVESTIGÁVEL EM PARTE"
    return (estado, "—")

print(checar({"existe": "Parcial", "retencao_dias": 14, "fora_do_host": False, "custodiante": "TI"}, 365))

## 2. Quanto tempo o log vive? Lendo o `logrotate`

A retenção é `rotate N` × o período (`daily`, `weekly`, `monthly`). A célula lê uma configuração de exemplo (o padrão do nginx no Ubuntu); se você estiver num Linux com nginx, ela tenta ler o arquivo de verdade primeiro.

In [ ]:
import re
from pathlib import Path

EXEMPLO_NGINX = """/var/log/nginx/*.log {
	daily
	missingok
	rotate 14
	compress
	delaycompress
	notifempty
	create 0640 www-data adm
	sharedscripts
}"""

def retencao_logrotate(texto):
    """Devolve (dias, explicação) a partir do texto de uma regra do logrotate."""
    periodo = {"daily": 1, "weekly": 7, "monthly": 30, "yearly": 365}
    dias_periodo = next((d for p, d in periodo.items() if re.search(rf"^\s*{p}\b", texto, re.M)), None)
    m = re.search(r"^\s*rotate\s+(\d+)", texto, re.M)
    if dias_periodo is None or m is None:
        return None, "faltou 'daily/weekly/...' ou 'rotate N': retenção não verificada"
    n = int(m.group(1))
    # o arquivo atual + N rotacionados: cerca de (N + 1) períodos de histórico
    return dias_periodo * (n + 1), f"{n} arquivos rotacionados × {dias_periodo} d + o arquivo atual"

real = Path("/etc/logrotate.d/nginx")
if real.is_file():
    texto, fonte = real.read_text(), str(real)
else:
    texto, fonte = EXEMPLO_NGINX, "exemplo (padrão do Ubuntu; NÃO verificado na máquina do cliente)"
dias, como = retencao_logrotate(texto)
print(f"Fonte: {fonte}\nRetenção: ~{dias} dias ({como})")

> 💡 Repare na palavra **"exemplo"** na saída. No mapa de evidências, uma retenção que veio daqui vai como *não verificada*. Ela só vira fato quando alguém lê o arquivo do servidor do cliente.

## 3. ➕ O mapa de evidências do canal "Denuncie um golpe"

As oito ameaças do Dia 16 (A01–A08), com o que o Juliano informou. Ajuste o que quiser no formulário; a retenção mínima é a que a sua razão justificar.

In [ ]:
#@title Mapa de evidências — canal "Denuncie um golpe" { display-mode: "form" }
retencao_minima_dias = 365  #@param {type:"integer"}
razao_retencao = "Denúncias viram inquérito; a polícia costuma pedir dados meses depois do fato."  #@param {type:"string"}

MAPA = [
    {"id": "A01", "stride": "S", "titulo": "'Denuncie' falso por SMS",
     "fonte": "Fora da cooperativa (SMS no celular do cooperado, operadora). Indiretas: denúncias de quem recebeu; retratos de DNS (Dia 13)",
     "registro": "Print do SMS falso com hora; linha do dns.stable.txt com o domínio novo",
     "existe": "Parcial", "retencao_dias": None, "fora_do_host": True, "custodiante": "Cooperado / operadora"},
    {"id": "A02", "stride": "I", "titulo": "Planilha semanal com CPF por e-mail",
     "fonte": "Log de envio do e-mail em nuvem da cooperativa",
     "registro": "Evento de envio: remetente, destinatário, assunto, nome e tamanho do anexo, hora",
     "existe": "Parcial", "retencao_dias": 30, "fora_do_host": True, "custodiante": "TI (informado, não verificado)"},
    {"id": "A03", "stride": "R", "titulo": "Sem log de auditoria de denúncias e exportações",
     "fonte": "Aplicação do canal (não existe)",
     "registro": "Seria: evento, protocolo, conta, src_ip, ts_utc, sha256_upload",
     "existe": "N", "retencao_dias": None, "fora_do_host": False, "custodiante": ""},
    {"id": "A04", "stride": "T", "titulo": "Print que não é imagem",
     "fonte": "Pasta de uploads em sa-web01 + nginx access log",
     "registro": "POST /denuncie com IP e hora; o arquivo em si. Falta: sha256 de cada upload no momento do recebimento",
     "existe": "Parcial", "retencao_dias": 15, "fora_do_host": False, "custodiante": "TI"},
    {"id": "A05", "stride": "D", "titulo": "Enxurrada de denúncias falsas",
     "fonte": "nginx access log (volume) + log de erro da aplicação/banco (impacto)",
     "registro": "Contagem de POST /denuncie por IP por minuto; erros 5xx e de disco no mesmo intervalo",
     "existe": "Parcial", "retencao_dias": 15, "fora_do_host": False, "custodiante": "TI"},
    {"id": "A06", "stride": "E", "titulo": "Área da ouvidoria protegida só por link",
     "fonte": "nginx access log",
     "registro": "GET /ouvidoria/* de IPs fora da rede interna; sem conta (falta auditoria)",
     "existe": "Parcial", "retencao_dias": 15, "fora_do_host": False, "custodiante": "TI"},
    {"id": "A07", "stride": "I", "titulo": "Chave do gateway de SMS vazada",
     "fonte": "Painel do gateway de SMS (terceiro)",
     "registro": "Histórico de envios por chave de API, com IP de origem e destinatários",
     "existe": "Parcial", "retencao_dias": None, "fora_do_host": True, "custodiante": "Fornecedor do gateway"},
    {"id": "A08", "stride": "S", "titulo": "Login de atendente por credential stuffing",
     "fonte": "Log de autenticação da intranet",
     "registro": "Falhas e sucesso por conta e por IP (o padrão do Dia 12)",
     "existe": "Parcial", "retencao_dias": 28, "fora_do_host": False, "custodiante": "TI"},
]

for l in MAPA:
    l["estado"], l["lacuna"] = checar(l, retencao_minima_dias)
    print(f"{l['id']} [{l['stride']}] {l['titulo']:<46} {l['estado']:<22} {l['lacuna']}")

## 4. O plano: no máximo cinco ações, ranqueadas

Cada ação diz quais ameaças ela torna investigáveis. O ranking é pelo número de ameaças fechadas; a do repúdio (A03) vai primeiro, como o checkpoint pede.

In [ ]:
ACOES = [
    {"acao": "Log de auditoria do canal (evento, protocolo, conta, src_ip, ts_utc, sha256_upload)",
     "fecha": ["A03", "A04", "A06", "A02"], "esforco_h": 8},
    {"acao": "Enviar nginx, auth e auditoria para um coletor fora de sa-web01, 12 meses",
     "fecha": ["A03", "A04", "A05", "A06", "A08"], "esforco_h": 6},
    {"acao": "Perguntar ao fornecedor do gateway: histórico por chave, IP de origem, retenção, exportação",
     "fecha": ["A07"], "esforco_h": 1},
    {"acao": "Confirmar retenção e campos do log de envio do e-mail em nuvem; tirar o CPF da planilha",
     "fecha": ["A02"], "esforco_h": 2},
    {"acao": "Incluir domínios '*denuncie*' na lista do dns-snapshot (Dia 13) e orientar a ouvidoria a guardar prints de SMS falsos",
     "fecha": ["A01"], "esforco_h": 1},
]
assert len(ACOES) <= 5, "O plano tem no máximo cinco ações"

# Regra do checkpoint: a primeira ação é a que CRIA o registro que fecha o repúdio (A03).
# As demais seguem pelo número de ameaças que tornam investigáveis.
cria_repudio = lambda a: "A03" in a["fecha"] and "auditoria do canal" in a["acao"]
ACOES.sort(key=lambda a: (not cria_repudio(a), -len(a["fecha"])))

for i, a in enumerate(ACOES, start=1):
    print(f"{i}. {a['acao']}\n   fecha: {', '.join(a['fecha'])} · ~{a['esforco_h']} h")

cobertas = sorted({t for a in ACOES for t in a["fecha"]})
print("\nAmeaças cobertas pelo plano:", ", ".join(cobertas))
print("Retenção proposta:", retencao_minima_dias, "dias —", razao_retencao)

## 5. O registro de auditoria, de verdade

Uma linha de exemplo do log proposto, com o hash do *upload* calculado no momento do recebimento e cada linha encadeada à anterior (a Inovação OSINT do README).

In [ ]:
import hashlib, json
from datetime import datetime, timezone

anterior = "0" * 64                                       # início da corrente

def auditar(evento, **campos):
    """Grava uma linha JSON com o hash da linha anterior; devolve o texto."""
    global anterior
    linha = {"evento": evento, "ts_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
             **campos, "hash_anterior": anterior}
    texto = json.dumps(linha, sort_keys=True, ensure_ascii=False)
    anterior = hashlib.sha256(texto.encode()).hexdigest()   # esta linha vira o elo da próxima
    return texto

print_falso = b"\x89PNG\r\n\x1a\n...(bytes do print)..."    # o arquivo recebido pelo formulário
log = [
    auditar("denuncia_recebida", protocolo="DN-000123", src_ip="198.51.100.61",
            sha256_upload=hashlib.sha256(print_falso).hexdigest(), tipo_real="PNG"),
    auditar("planilha_exportada", conta="atendente02", src_ip="10.20.40.12", linhas=37),
]
Path("notas").mkdir(exist_ok=True)
Path("notas/isca-dia17-auditoria-exemplo.jsonl").write_text("\n".join(log) + "\n", encoding="utf-8")
for l in log:
    print(l)
print("\nÚltimo hash (guarde FORA do servidor):", anterior)

In [ ]:
# Teste: alguém apaga a linha da exportação e reescreve só a primeira. A corrente denuncia?
def verificar(linhas, ultimo_hash_guardado):
    elo = "0" * 64
    for n, texto in enumerate(linhas, start=1):
        if json.loads(texto)["hash_anterior"] != elo:
            return f"❌ quebra na linha {n}"
        elo = hashlib.sha256(texto.encode()).hexdigest()
    return "✅ íntegro" if elo == ultimo_hash_guardado else "❌ o fim da corrente não bate com o hash guardado fora"

print("Log original:          ", verificar(log, anterior))
print("Sem a linha da planilha:", verificar(log[:1], anterior))

## 6. 🎮 Quiz: os quatro fatos

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("'A evidência está nos logs.' Isso preenche a coluna 'registro específico'?", ["a) Sim", "b) Não: precisa do evento, campo ou padrão de linha"], "b"),
    ("O nginx mostra GET /admin/export e um IP. O que falta para o repúdio?", ["a) Nada", "b) A conta logada, que só a aplicação conhece"], "b"),
    ("auth.log no próprio servidor invadido é resistente a adulteração?", ["a) Sim, se tiver permissão 640", "b) Não: o root pode editá-lo; precisa ir para fora do host"], "b"),
    ("Você não achou a retenção do log de API do provedor. Escreve...", ["a) 'provavelmente 30 dias'", "b) 'desconhecida, perguntar ao provedor' (e isso é um achado)"], "b"),
    ("Qual é a primeira ação do plano?", ["a) A mais barata", "b) A que fecha a lacuna de repúdio"], "b"),
    ("Retenção proposta para logs com IP de cooperados:", ["a) para sempre, por segurança", "b) um prazo com razão escrita ligada à finalidade"], "b"),
]
acertos = 0
for i, (p, ops, certa) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {p}")
    for o in ops:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b"):
            r = input("   Resposta (a/b): ").strip().lower()
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ Era {certa}.")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 5 else "— Meta: 5.")

## 7. 📝 Mapa e plano em arquivos

In [ ]:
import csv
with open("notas/isca-dia17-mapa-evidencias.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["ID", "STRIDE", "Ameaça", "Fonte", "Registro específico", "Existe hoje?", "Retenção hoje (dias)",
                "Fora do host?", "Custodiante", "Estado", "Lacuna e correção"])
    for l in MAPA:
        w.writerow([l["id"], l["stride"], l["titulo"], l["fonte"], l["registro"], l["existe"],
                    l["retencao_dias"] if l["retencao_dias"] is not None else "desconhecida",
                    "sim" if l["fora_do_host"] else "não", l["custodiante"] or "?", l["estado"], l["lacuna"]])

plano = ["# Plano de prontidão — canal 'Denuncie um golpe' (Dia 17)", ""]
for i, a in enumerate(ACOES, start=1):
    plano.append(f"{i}. **{a['acao']}** — torna investigáveis: {', '.join(a['fecha'])}; "
                 f"retenção {retencao_minima_dias} dias ({razao_retencao}); esforço ~{a['esforco_h']} h.")
plano += ["", "## Se o canal for atacado no primeiro mês",
          "Hoje conseguiríamos investigar só em parte A04, A05, A06 e A08 (nginx e auth.log, 15–28 dias, no próprio servidor). "
          "Não conseguiríamos investigar A03 (não há auditoria) nem dizer quem exportou a planilha. "
          "Ao fornecedor do gateway de SMS pediríamos primeiro o histórico de envios por chave de API, com IP de origem, e a retenção dele."]
Path("notas/isca-dia17-plano-prontidao.md").write_text("\n".join(plano) + "\n", encoding="utf-8")
print("\n".join(plano))
try:
    from google.colab import files
    for f in ("isca-dia17-mapa-evidencias.csv", "isca-dia17-plano-prontidao.md"):
        files.download(f"notas/{f}")
except ImportError:
    print("\n(Fora do Colab: os arquivos ficaram em notas/)")

## ✅ Antes de fechar o Dia 17

- [ ] Quiz com pelo menos 5/6
- [ ] `notas/isca-dia17-mapa-evidencias.csv` com as 8 ameaças
- [ ] `notas/isca-dia17-plano-prontidao.md` com até 5 ações, a primeira fechando a A03
- [ ] `day17-evidence-map`, `day17-evidence-dfd.png` e `day17-readiness-plan.md` do original (README, Passos 1 a 3)
- [ ] `status` do dia 17 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 18):** montar o seu laboratório de investigação isolado e mapear a topologia. É o fechamento da Fase 1, e a Vigia quer um lugar seguro para abrir o IscaKit.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>